<div align="center">

# Week 11: Security of Cyber-Physical Systems: False Data Injection and Defences

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-11) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/Week11_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also loads the spring day of Week 1 and the thirteen meters of the control room of Week 6. The constants of the island of Weeks 2, 5 and 8 follow. The function `lab_rng` draws the same random numbers as the interactive lab. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import hashlib
import hmac
import io
import itertools
import math
import struct
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

def lab_rng(seed):
    """The random generator of the interactive lab, so that the Colab notebook draws exactly the same numbers."""
    s = seed & 0xFFFFFFFF
    def uniform():
        nonlocal s
        s = (s + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((s ^ (s >> 15)) * (s | 1)) & 0xFFFFFFFF
        t = (t ^ ((t + (((t ^ (t >> 7)) * (t | 61)) & 0xFFFFFFFF)) & 0xFFFFFFFF)) & 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    def normal():
        u = v = 0.0
        while u == 0.0:
            u = uniform()
        while v == 0.0:
            v = uniform()
        return math.sqrt(-2 * math.log(u)) * math.cos(2 * math.pi * v)
    return uniform, normal

# The spring weekday of Week 1 on the campus, one value per hour (kW, m/s, W/m2 and degrees Celsius)
HOURS = np.arange(24)
LOAD_KW = np.array([262, 255, 250, 248, 252, 270, 330, 430, 520, 570, 585, 590,
                    575, 580, 585, 575, 560, 530, 520, 500, 450, 390, 320, 280], float)
EV_KW = np.array([0, 0, 0, 0, 0, 0, 0, 22, 66, 66, 44, 22,
                  22, 22, 22, 44, 66, 110, 110, 88, 44, 22, 0, 0], float)
WIND_MS = np.array([7.8, 8.0, 8.2, 8.0, 7.6, 7.0, 6.2, 5.4, 4.8, 4.4, 4.2, 4.6,
                    5.0, 5.2, 5.0, 4.6, 4.2, 3.8, 3.6, 3.5, 3.8, 4.6, 5.8, 7.0])
G_SKY = 950 * np.maximum(0, np.sin(np.pi * (HOURS + 0.5 - 6) / 13))
T_AMB = 12 + 7 * np.sin(np.pi * (HOURS - 9) / 12)
PV_KW = 400 * G_SKY / 1000 * (1 - 0.004 * (T_AMB + 0.03 * G_SKY - 25))
WIND_KW = np.where(WIND_MS < 3, 0.0, np.where(WIND_MS < 12, 250 * (WIND_MS ** 3 - 27) / (12 ** 3 - 27), 250.0))
NET_KW = LOAD_KW + EV_KW - PV_KW - WIND_KW          # the import without the battery
LIMIT_KW = 600.0                                     # the contracted import limit of the campus
# The thirteen meters of the control room at 19:00 (Week 6): six feeders, six buildings and the main meter
NAMES = ["B1 feeder", "B2 feeder", "B3 feeder", "B4 feeder", "B5 feeder", "Car park feeder",
         "B1 building", "B2 building", "B3 building", "B4 building", "B5 building", "Car park chargers", "Main meter"]
H_SE = np.vstack([np.eye(6), np.eye(6), np.ones((1, 6))])
SIGMA_SE = np.array([1.5] * 6 + [2.5] * 6 + [3.0])
Z_SE = np.array([130.2, 88.4, 103.3, 64.1, 115.8, 90.0, 130.2, 89.5, 107.9, 65.7, 114.7, 85.3, 585.5])
# The island of Weeks 2, 5 and 8: inertia M, damping D and droop gain K of the battery
M_KWS_HZ, D_KW_HZ, K_KW_HZ = 80.0, 12.0, 125.0
print("Setup complete. Running in Colab:", IN_COLAB, "| meters:", len(NAMES), "| island M, D, K =",
      M_KWS_HZ, D_KW_HZ, K_KW_HZ)

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 11.1": (lambda: len(hashlib.sha256(b'').hexdigest()), 1e-9),
    "Exercise 11.2": (lambda: bin(int.from_bytes(hashlib.sha256(bytes.fromhex('02 06 00 64 FF 06')).digest(), 'big') ^ int.from_bytes(hashlib.sha256(bytes.fromhex('02 06 00 64 FF 07')).digest(), 'big')).count('1'), 1e-9),
    "Exercise 11.3": (lambda: 4, 1e-9),
    "Exercise A": (lambda: float(NET_KW[18] + 200), 1e-6),
    "Exercise B": (lambda: wls(Z_SE + injection({'B3 feeder': 50, 'B3 building': 50}))['J'], 1e-6),
    "Exercise C": (lambda: island(5, 20)['alarm'], 1e-6),
    "Exercise D": (lambda: sum(12 in P for P in SECURE6), 1e-9),
    "Exercise E": (lambda: STAGE11['largest hidden shift of B3, B3 feeder meter protected (kW)'], 1e-6),
}

## Python step 11: Hashes and keys

A message that a controller obeys must be genuine. This step computes the fingerprint of a Modbus frame with a hash function. It adds a secret key to obtain a tag that only the holders of the key can compute, and it numbers the messages against replay &#91;[2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "Z_SE" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-11/"
               "NB11_cps_security.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Hash functions

In [ ]:
# Python practice: a small example for learning the language
frame = bytes.fromhex("02 06 00 64 FF 06")     # a Modbus request: device 2, register 100, value -250
digest = hashlib.sha256(frame).hexdigest()        # the fingerprint: 32 bytes, written as 64 hexadecimal digits
print("SHA-256 of the frame:      ", digest)
changed = bytes.fromhex("02 06 00 64 FF 07")      # the last bit flipped
print("SHA-256 after one bit:     ", hashlib.sha256(changed).hexdigest())

A hash function turns bytes of any length into a fingerprint of fixed length. A function of the family Secure Hash Algorithm (SHA), SHA-256, gives 32 bytes. One changed bit changes about half of the bits of the fingerprint, and nobody knows how to find a second message with the same fingerprint. Anyone can compute a fingerprint, however, because it uses no secret.

### A keyed hash

In [ ]:
# Python practice: a small example for learning the language
KEY = bytes.fromhex("5f1c9a3e7b2d4c6f8e0a1b3c5d7e9f20")    # a secret of 16 bytes: controller and inverter
tag = hmac.new(KEY, frame, hashlib.sha256).digest()[:8]        # the first 8 bytes of the keyed hash
print("tag with the key:          ", tag.hex(" ").upper())
guess = hmac.new(b"not the right key", frame, hashlib.sha256).digest()[:8]
print("tag with another key:      ", guess.hex(" ").upper())
print("equal:", hmac.compare_digest(tag, guess))

The hash-based message authentication code (HMAC) mixes a secret key into the hash &#91;[11](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#ref-11)&#93;. Only a holder of the key can compute the tag of a message. A correct tag therefore shows that the message comes from a holder of the key and arrived unchanged. The function `hmac.compare_digest` compares two tags in a time that does not depend on the position of the first difference. The time of the answer then reveals nothing about the tag.

### Counters against replay

In [ ]:
# Python practice: a small example for learning the language
def accept(counter, last):
    """The receiver takes a message only if its counter is larger than the last one it accepted."""
    return counter > last

last = 0
for counter in [1, 2, 3, 2, 4]:
    ok = accept(counter, last)
    print(f"counter {counter}: " + ("accepted" if ok else "rejected, an old counter: a replay"))
    if ok:
        last = counter

A tag alone does not stop a replay: An old message with its old tag is still genuine. The sender therefore numbers its messages, and the number is part of the bytes under the tag. The receiver keeps the last number that it accepted and rejects every message that is not newer.

**Quick check.** Why can the cyclic redundancy check (CRC) of Week 8 not protect a frame against an attacker?

<details><summary>Answer</summary>

The CRC uses no secret. An attacker who changes a frame computes the new CRC as easily as the sender did. The CRC finds random errors on the line, not intent.

</details>

### Exercises for Python step 11

**Exercise 11.1.** Store the number of hexadecimal digits of a SHA-256 fingerprint in `n_111`.

In [ ]:
n_111 = None   # your answer

check("Exercise 11.1", n_111)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_111 = len(hashlib.sha256(b"any message").hexdigest())

check("Exercise 11.1", n_111)

**Exercise 11.2.** Store the number of bits in which the SHA-256 fingerprints of `frame` and `changed` differ in `d_112`.

In [ ]:
d_112 = None   # your answer

check("Exercise 11.2", d_112)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
a = int.from_bytes(hashlib.sha256(frame).digest(), "big")
b = int.from_bytes(hashlib.sha256(changed).digest(), "big")
d_112 = bin(a ^ b).count("1")

check("Exercise 11.2", d_112)

**Exercise 11.3.** A receiver starts with `last = 4` and gets the counters 5, 6, 6, 9, 7 and 10 in this order. Store the number of accepted messages in `n_113`.

In [ ]:
n_113 = None   # your answer

check("Exercise 11.3", n_113)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
last, n_113 = 4, 0
for counter in [5, 6, 6, 9, 7, 10]:
    if accept(counter, last):
        last, n_113 = counter, n_113 + 1

check("Exercise 11.3", n_113)

## 1. Threats: A forged command on the bus &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#ref-1)&#93;

*Lecture: [Threats and attack surfaces](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#threats)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "accept" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-11/"
               "NB11_cps_security.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The battery inverter of the campus obeys every Modbus frame whose cyclic redundancy check (CRC) is right. Python builds the frame of an attacker who sets the battery to charge with full power at 19:00. It checks the frame as the inverter does and computes what the command does to the import of the campus.

**From the formulas to the code.** Function 06, Write Single Register, carries the address of the device, the function code, the register and a value of 16 bits. The CRC follows: `write_request(address, register, value)`. The import is $g_t = n_t - p_t$ with the battery power $p_t$, positive when the battery discharges. The rule of Week 1 discharges $p_t = \min(\max(0, n_t - 450), 250)$ kW.

In [ ]:
def crc16_modbus(data):
    """CRC-16 of Modbus RTU (Week 8): initial value 0xFFFF, reflected polynomial 0xA001."""
    crc = 0xFFFF
    for b in data:
        crc ^= b
        for _ in range(8):
            crc = (crc >> 1) ^ 0xA001 if crc & 1 else crc >> 1
    return crc

def write_request(address, register, value):
    """Function 06, Write Single Register, with a signed value of 16 bits and the CRC, low byte first."""
    pdu = bytes([address, 0x06]) + struct.pack(">Hh", register, value)
    crc = crc16_modbus(pdu)
    return pdu + bytes([crc & 0xFF, crc >> 8])

RULE_KW = np.clip(NET_KW - 450.0, 0.0, 250.0)            # the rule of Week 1: discharge what lies above 450 kW
forged = write_request(2, 100, -250)                     # inverter 2, setpoint register 100: charge with 250 kW
print("forged frame:", forged.hex(" ").upper(), "| CRC at the inverter:", hex(crc16_modbus(forged)), "-> accepted")
BAT_FORGED = RULE_KW.copy()
BAT_FORGED[19] = -250.0                                  # the inverter obeys from 19:00 to 20:00
for hour in range(17, 22):
    print(f"{hour}:00  import with the rule {NET_KW[hour] - RULE_KW[hour]:7.2f} kW | "
          f"with the forged command {NET_KW[hour] - BAT_FORGED[hour]:7.2f} kW")
excess = NET_KW[19] - BAT_FORGED[19] - LIMIT_KW
print(f"at 19:00 the campus draws {excess:.2f} kW more than its contracted limit of {LIMIT_KW:.0f} kW")

**Python note.** The call `struct.pack(">Hh", register, value)` packs the register as an unsigned and the value as a signed number of 16 bits, so −250 becomes the two bytes `FF 06`. The CRC of a frame that includes its own CRC is 0, which the inverter reads as an intact frame.

**Exercise A.** The attacker sends a command of −200 kW for 18:00 instead. Store the import at 18:00 in kW in `imp_a`.

In [ ]:
imp_a = None   # your answer

check("Exercise A", imp_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
imp_a = NET_KW[18] + 200

check("Exercise A", imp_a)

In [ ]:
# Figure: the import of the spring day with the rule of Week 1 and with the forged command
fig, ax = plt.subplots(figsize=(8, 3.4))
hrs = np.arange(25)
ext = lambda a: np.append(a, a[-1])
ax.step(hrs, ext(NET_KW), where="post", color="#56616D", label="without the battery")
ax.step(hrs, ext(NET_KW - RULE_KW), where="post", color="#1E6FA8", lw=2, label="with the rule of Week 1")
ax.step(hrs, ext(NET_KW - BAT_FORGED), where="post", color="#B53F2E", ls="--", label="with the forged command at 19:00")
ax.axhline(LIMIT_KW, color="#B53F2E", lw=0.8, ls=":")
ax.text(0.5, LIMIT_KW + 12, "contracted limit 600 kW", fontsize=8, color="#B53F2E")
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 3))
ax.set_xlabel("hour of the spring day")
ax.set_ylabel("import (kW)")
ax.legend(fontsize=8, loc="upper left")
plt.show()

**Observe and reflect.** Which entry points of the campus could carry this frame to the inverter? Which of them need physical access, and which can be reached over the network?

## 2. False data injection: Your attacks in the switchboard &#91;[8](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#ref-8)&#93;

*Lecture: [False data injection](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#false-data-injection)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "write_request" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-11/"
               "NB11_cps_security.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The control room estimates the six feeders of the campus from thirteen meters, as in Week 6. It sheds the dormitory B3 when its estimate exceeds 150 kW. Python plays the attacker. A change of one meter fails the chi-square test, while an injection along a column of $H$ passes it.

**From the formulas to the code.** The estimate $\hat{\mathbf{x}} = (H^T W H)^{-1} H^T W \mathbf{z}$, the residual $\mathbf{r} = \mathbf{z} - H\hat{\mathbf{x}}$ and the test value $J = \mathbf{r}^T W \mathbf{r}$ come from `wls(z)`. The threshold is the 99 percent quantile of the chi-square distribution with $13 - 6 = 7$ degrees of freedom. A hidden injection $\mathbf{a} = H\mathbf{c}$ moves the estimate by $\mathbf{c}$ and leaves $J$ unchanged.

In [ ]:
W_SE = np.diag(1 / SIGMA_SE ** 2)
G_SE = H_SE.T @ W_SE @ H_SE
THRESHOLD = float(chi2.ppf(0.99, 13 - 6))                # the chi-square test of Week 6
SHED_KW = 150.0                                          # the control room sheds B3 above this estimate

def wls(z):
    """Weighted least squares (Week 6): the six feeder estimates, the residuals and the test value J."""
    x = np.linalg.solve(G_SE, H_SE.T @ W_SE @ z)
    r = z - H_SE @ x
    return dict(x=x, r=r, J=float(r @ W_SE @ r))

def injection(changes):
    """The vector a of the attacker from a dictionary of meter names and changes in kW."""
    a = np.zeros(len(NAMES))
    for name, kw in changes.items():
        a[NAMES.index(name)] = kw
    return a

base = wls(Z_SE)
print(f"no attack:     B3 estimated at {base['x'][2]:.2f} kW, J = {base['J']:.2f} against the threshold {THRESHOLD:.2f}")
gain = np.linalg.solve(G_SE, H_SE.T @ W_SE)[2, 2]       # the share of a change of the B3 feeder meter in the estimate
need = (SHED_KW - base["x"][2]) / gain
one = wls(Z_SE + injection({"B3 feeder": need}))
print(f"one meter:     the B3 feeder meter must read {need:.2f} kW more, J = {one['J']:.2f} -> bad-data alarm")
c = np.zeros(6)
c[2] = 50.0                                              # the attacker wants the estimate of B3 50 kW higher
a = H_SE @ c                                             # a hidden injection along the third column of H
hidden = wls(Z_SE + a)
print("hidden attack: changes", [NAMES[i] for i in np.flatnonzero(a)])
print(f"hidden attack: B3 estimated at {hidden['x'][2]:.2f} kW, J = {hidden['J']:.2f} -> no alarm, B3 is shed")

**Python note.** The expression `np.linalg.solve(G_SE, H_SE.T @ W_SE)` is the matrix that turns readings into estimates. Its element `[2, 2]` is the share of a change of the B3 feeder meter that reaches the estimate of B3. The call `np.flatnonzero(a)` lists the meters that the hidden injection changes.

**Exercise B.** The attacker adds 50 kW to the B3 feeder and the B3 building meters but forgets the main meter. Store the test value J in `j_b`.

In [ ]:
j_b = None   # your answer

check("Exercise B", j_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
j_b = wls(Z_SE + injection({"B3 feeder": 50, "B3 building": 50}))["J"]

check("Exercise B", j_b)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lab.html) opens the main switchboard of the campus with the laptop of an attacker. Paste the copied rows into `MY_ATTACKS`, and the code sends each of your attacks to the control room again.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_ATTACKS = """
"""
def parse(text):
    """'B3 feeder=+50;Main meter=+50' -> {'B3 feeder': 50.0, 'Main meter': 50.0}; 'none' -> {}."""
    if not isinstance(text, str) or text.strip() in ("", "none"):
        return {}
    return {k.strip(): float(v) for k, v in (p.split("=") for p in text.split(";"))}

def names_of(text):
    """'B1 feeder;B2 feeder' -> ['B1 feeder', 'B2 feeder']; 'none' -> []."""
    if not isinstance(text, str) or text.strip() in ("", "none"):
        return []
    return [p.strip() for p in text.split(";")]

def control_room(changes, protected):
    """An authentication alarm if a protected meter was changed, else the chi-square test and the B3 estimate."""
    if any(name in protected for name in changes):
        return "authentication", float("nan"), float("nan")
    res = wls(Z_SE + injection(changes))
    return ("bad data" if res["J"] > THRESHOLD else "none"), round(res["J"], 2), round(float(res["x"][2]), 2)

if MY_ATTACKS.strip():
    runs = pd.read_csv(io.StringIO(MY_ATTACKS.strip()))
    source = "your own attacks in the switchboard"
else:
    hide = "B3 feeder=+50;B3 building=+50;Main meter=+50"
    runs = pd.DataFrame({"run": [1, 2, 3], "changes": ["B3 feeder=+50", hide, hide],
                         "protected": ["none", "none", ";".join(NAMES[:6])],
                         "alarm": ["bad data", "none", "authentication"]})
    source = "a simulated student"
out = [control_room(parse(r.changes), names_of(r.protected)) for r in runs.itertuples()]
runs["model_alarm"], runs["model_J"], runs["model_B3_kw"] = zip(*out)
print("Data:", source)
print(runs[["run", "alarm", "model_alarm", "model_J", "model_B3_kw"]].to_string(index=False))

In [ ]:
# Figure: the six feeder estimates without an attack, with one meter changed and with the hidden attack
fig, ax = plt.subplots(figsize=(9, 3.4))
cases = [("no attack", base, "#9DB7CC"), (f"one meter, +{need:.2f} kW", one, "#C98510"),
         ("hidden attack, +50 kW", hidden, "#B53F2E")]
for k, (name, res, col) in enumerate(cases):
    ax.bar(np.arange(6) + (k - 1) * 0.27, res["x"], 0.27, color=col, label=f"{name}: J = {res['J']:.2f}")
ax.axhline(SHED_KW, color="#B53F2E", ls=":", lw=1)
ax.text(2.45, SHED_KW + 5, "B3 shed above 150 kW", fontsize=8, color="#B53F2E")
ax.set_xticks(range(6))
ax.set_xticklabels(["B1", "B2", "B3", "B4", "B5", "Car park"])
ax.set_ylim(0, 200)
ax.set_ylabel("estimate at 19:00 (kW)")
ax.legend(fontsize=8, loc="upper right")
plt.show()

**Observe and reflect.** Why does the hidden attack also change the main meter? What would the attacker have to change if the main meter could not be touched?

## 3. Replay: The watermark of the battery &#91;[9](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#ref-9)&#93;

*Lecture: [Attacks on control loops: Replay and denial of service](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#replay)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "wls" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-11/"
               "NB11_cps_security.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** In the island mode of the campus, the battery holds the frequency with its droop of Weeks 2 and 5. An attacker records the frequency measurement for 40 s and then replays it to the battery controller. Twenty seconds later a forged command starts three cars with 66 kW. Python simulates the island with and without a secret watermark on the battery power and detects the replay.

**From the formulas to the code.** The frequency follows $f_{t+1} = \alpha f_t + \beta (P_t - d_t)$ in steps of $\Delta t = 0.1$ s, with $\alpha = e^{-D\Delta t/M}$ = `ALPHA` and $\beta = (1 - \alpha)/D$ = `BETA`. The battery sets $P_t = -K y_t + \omega_t$ from the measurement $y_t$ and the watermark $\omega_t$. The detector $g$ is the slope of the movement $m_{t+1} = y_{t+1} - (\alpha - \beta K)\,y_t$ against $\omega_t$ over the last $N$ steps, divided by $\beta$. It alarms when $g < 0.5$, and the battery then switches to its own measurement. The function `island(watermark_kw, window_s)` runs the 100 s.

In [ ]:
DT = 0.1                                             # the frequency meter reports every 0.1 s
ALPHA = math.exp(-D_KW_HZ * DT / M_KWS_HZ)           # the share of a deviation that remains after one step
BETA = (1 - ALPHA) / D_KW_HZ                         # Hz of change per kW of imbalance in one step
RHO = ALPHA - BETA * K_KW_HZ                         # one step with the droop of the battery
SIGMA_V, SIGMA_D = 0.010, 2.0                        # noise of the meter (Hz) and random load changes (kW)
REC, REP, STRIKE, END = 200, 600, 800, 1000          # steps: recording from 20 s, replay from 60 s, cars at 80 s

def island(watermark_kw, window_s, attack=True, seed=11, threshold=0.5):
    """100 s of the island: a replay of the frequency from 60 s and three cars of 66 kW from 80 s."""
    _, normal = lab_rng(seed)
    n = int(round(window_s / DT))
    f = 0.0
    measured, seen, omega, freq = (np.zeros(END) for _ in range(4))
    g = np.full(END, np.nan)
    alarm = trip = None
    for t in range(END):
        w, d, v = watermark_kw * normal(), SIGMA_D * normal(), SIGMA_V * normal()
        measured[t] = f + v                                       # what the frequency meter measures
        replaying = attack and t >= REP and alarm is None
        seen[t] = measured[t - (REP - REC)] if replaying else measured[t]
        omega[t], freq[t] = w, f
        if watermark_kw > 0 and alarm is None and t > n:          # the test over the last n steps
            k = np.arange(t - n, t)
            m = seen[k + 1] - RHO * seen[k]                       # the movement that the droop does not explain
            g[t] = (omega[k] @ m) / (BETA * (omega[k] @ omega[k]))
            if g[t] < threshold:
                alarm = round(t * DT, 1)                          # from now on the battery uses its own measurement
        p = -K_KW_HZ * seen[t] + w
        load = d + (66.0 if attack and t >= STRIKE else 0.0)
        f = ALPHA * f + BETA * (p - load)
        if abs(f) > 2.5:                                          # the under-frequency protection of Week 2
            trip = round((t + 1) * DT, 1)
            break
    jitter = round(1000 * float(np.std(freq[100:REP])), 2)       # mHz in normal operation, 10 s to 60 s
    return dict(alarm=alarm, false_alarm=alarm is not None and alarm < REP * DT, trip=trip, jitter_mhz=jitter,
                f=freq[:t + 1], seen=seen[:t + 1], g=g[:t + 1])

print(f"alpha = {ALPHA:.4f}, beta = {1000 * BETA:.4f} mHz per kW, alpha - beta K = {RHO:.4f}")
for wm, win in [(0, 10), (10, 5), (5, 10), (5, 20), (2, 20), (2, 30)]:
    r = island(wm, win)
    state = f"tripped at {r['trip']} s" if r["trip"] else f"held at {r['f'][-1]:+.2f} Hz"
    print(f"watermark {wm:>2} kW, window {win:>2} s: alarm {r['alarm']} s"
          + (" (false alarm)" if r["false_alarm"] else "") + f" | island {state} | jitter {r['jitter_mhz']} mHz")

**Python note.** The function `lab_rng` draws the same random numbers as the interactive lab, in the order watermark, load and meter. Every run therefore meets the same noise, and only the size of the watermark changes. The product `omega[k] @ m` is the sum of the products, the numerator of a slope through the origin.

**Exercise C.** Store the time of the alarm in seconds with a watermark of 5 kW and a window of 20 s in `t_c`.

In [ ]:
t_c = None   # your answer

check("Exercise C", t_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_c = island(5, 20)["alarm"]

check("Exercise C", t_c)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lab.html) runs the replay at the island controller of the campus. Paste the copied rows into `MY_RUNS`, and the code repeats each of your runs.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_RUNS = """
"""
if MY_RUNS.strip():
    runs = pd.read_csv(io.StringIO(MY_RUNS.strip()))
    source = "your own runs at the island controller"
else:
    runs = pd.DataFrame({"run": [1, 2, 3, 4], "watermark_kw": [0, 10, 2, 2], "window_s": [10, 5, 20, 30],
                         "alarm_s": [None, 62.6, 22.0, 72.7]})
    source = "a simulated student"
model = [island(r.watermark_kw, r.window_s) for r in runs.itertuples()]
runs["model_alarm_s"] = [m["alarm"] for m in model]
runs["model_trip_s"] = [m["trip"] for m in model]
print("Data:", source)
print(runs[["run", "watermark_kw", "window_s", "alarm_s", "model_alarm_s", "model_trip_s"]].to_string(index=False))

In [ ]:
# Figure: the replay without a watermark and with a watermark of 2 kW, and the detector g
plain, marked, fast = island(0, 10), island(2, 30), island(10, 5)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
secs = lambda r: np.arange(len(r["f"])) * DT
ax[0].plot(secs(plain), plain["f"], color="#B53F2E", lw=1.2, label="no watermark: true frequency")
ax[0].plot(secs(plain), plain["seen"], color="#C98510", lw=0.8, label="no watermark: what the battery sees")
ax[0].plot(secs(marked), marked["f"], color="#1E6FA8", lw=1.2, label="watermark 2 kW, window 30 s")
for x, txt in ((60, "replay"), (80, "three cars")):
    ax[0].axvline(x, color="#56616D", ls=":", lw=1)
    ax[0].text(x + 0.8, 0.12, txt, fontsize=8)
ax[0].set_ylim(-2.6, 0.3)
ax[0].set_xlabel("time (s)")
ax[0].set_ylabel("frequency deviation (Hz)")
ax[0].legend(fontsize=7, loc="lower left")
for r, col, setting in ((marked, "#1E6FA8", "2 kW, 30 s"), (fast, "#2E8B57", "10 kW, 5 s")):
    ax[1].plot(secs(r), r["g"], color=col, lw=1, label=f"g, watermark {setting}")
ax[1].axhline(0.5, color="#B53F2E", ls="--", lw=1)
ax[1].axvline(60, color="#56616D", ls=":", lw=1)
ax[1].set_xlim(0, 100)
ax[1].set_xlabel("time (s)")
ax[1].set_ylabel("response to the watermark g")
ax[1].legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

**Observe and reflect.** Why does the replay without a watermark raise no alarm at all? Why does a smaller watermark need a longer window?

## 4. Defences: Tags, counters and protected meters &#91;[12](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#ref-12)&#93;

*Lecture: [Defences in depth](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#defences)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "island" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-11/"
               "NB11_cps_security.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Two defences answer the attacks of Sections 1 and 2. A tag and a counter make the inverter reject forged and replayed frames, at the price of a longer poll cycle. Protected meters take the hidden directions away from the attacker of the state estimate. Python tests both.

**From the formulas to the code.** The frame carries a counter and the tag $\text{HMAC}_K(\text{frame}\,\|\,\text{counter})$, cut to 8 bytes, in front of the CRC: `secure_request`. `Inverter.receive` checks the CRC, the tag and the counter. A hidden injection must leave the protected meters $P$ alone, $H_P\mathbf{c} = 0$. No $\mathbf{c} \neq 0$ solves this when the rank of $H_P$ equals 6: `hidden_possible(P)`.

In [ ]:
KEY = bytes.fromhex("5f1c9a3e7b2d4c6f8e0a1b3c5d7e9f20")     # shared by the battery controller and the inverter

def secure_request(key, counter, address, register, value):
    """Function 06 with a counter of 4 bytes and a tag of 8 bytes (HMAC with SHA-256) in front of the CRC."""
    body = bytes([address, 0x06]) + struct.pack(">HhI", register, value, counter)
    body += hmac.new(key, body, hashlib.sha256).digest()[:8]
    crc = crc16_modbus(body)
    return body + bytes([crc & 0xFF, crc >> 8])

class Inverter:
    """Accepts a frame only if its CRC, its tag and its counter are all right."""
    def __init__(self, key):
        self.key, self.last, self.setpoint = key, 0, 0
    def receive(self, frame):
        if crc16_modbus(frame) != 0:
            return "rejected: damaged frame"
        body, tag = frame[:-10], frame[-10:-2]
        if not hmac.compare_digest(hmac.new(self.key, body, hashlib.sha256).digest()[:8], tag):
            return "rejected: wrong tag"
        register, value, counter = struct.unpack(">HhI", body[2:10])
        if counter <= self.last:
            return "rejected: old counter"
        self.last, self.setpoint = counter, value
        return f"accepted: setpoint {value} kW"

def with_crc(body):
    crc = crc16_modbus(body)
    return body + bytes([crc & 0xFF, crc >> 8])

inverter = Inverter(KEY)
genuine = secure_request(KEY, 41, 2, 100, 120)                # the controller: discharge with 120 kW
print(len(genuine), "bytes:", genuine.hex(" ").upper())
replies = {"genuine frame": inverter.receive(genuine)}
fake = bytearray(genuine[:-2])
fake[4:6] = struct.pack(">h", -250)                           # the attacker changes the value and repairs the CRC
replies["forged frame"] = inverter.receive(with_crc(bytes(fake)))
replies["replayed frame"] = inverter.receive(genuine)
for kind, reply in replies.items():
    print(f"{kind:<15} {reply}")
rng = np.random.default_rng(11)
fake[6:10] = struct.pack(">I", 42)                            # a new counter, and random tags
guessed = sum(inverter.receive(with_crc(bytes(fake[:10]) + rng.bytes(8))).startswith("accepted") for _ in range(2000))
print("random tags accepted:", guessed, "of 2000")
ATTACKER_ACCEPTED = guessed + sum(replies[k].startswith("accepted") for k in ("forged frame", "replayed frame"))

def poll_time(n_reg, baud, extra=0, t_resp=5.0):
    """One Modbus transaction in ms (Week 8); extra bytes in the request and in the response."""
    t_c = 11 / baud * 1000
    t35 = 1.75 if baud > 19200 else 3.5 * t_c
    return (8 + 5 + 2 * n_reg + 2 * extra) * t_c + 2 * t35 + t_resp

BUS = [10, 20, 40]                                            # registers of the meter, the inverter and the BMS
PLAIN_MS = sum(poll_time(n, 19200) for n in BUS)
SECURE_MS = sum(poll_time(n, 19200, extra=12) for n in BUS)
print(f"poll cycle at 19200 baud: {PLAIN_MS:.2f} ms without, {SECURE_MS:.2f} ms with counter and tag")

def hidden_possible(protected):
    """True if a hidden injection a = H c with c not 0 leaves every protected meter unchanged."""
    rows = H_SE[list(protected)]
    return len(rows) == 0 or np.linalg.matrix_rank(rows) < 6

print("six feeder meters protected, hidden attack possible:", hidden_possible(range(6)))
print("main meter and four feeder meters protected, hidden attack possible:", hidden_possible([12, 0, 1, 2, 3]))
SECURE6 = [P for P in itertools.combinations(range(13), 6) if not hidden_possible(P)]
print(len(SECURE6), "of", math.comb(13, 6), "sets of six meters leave no hidden attack")
OMEGA = SIGMA_SE ** 2 - np.diag(H_SE @ np.linalg.solve(G_SE, H_SE.T))   # the variance of each residual (Week 6)
for i in (2, 8):
    shift = base["r"][i] + math.sqrt(THRESHOLD * OMEGA[i])
    print(f"{NAMES[i]} meter protected: the largest shift of B3 that passes the test is {shift:.2f} kW")

**Python note.** The tag covers the address, the function, the register, the value and the counter. A change of any of them, the counter of an old frame included, makes the tag wrong. A random tag of 64 bits is right with a chance of $2^{-64}$, about $5.4 \times 10^{-20}$. With one meter of B3 protected, every shift of the B3 estimate shows up in the residual of that meter. The array `OMEGA` holds the variance of each residual, and the test lets that residual grow to $\sqrt{J_{max}\,\Omega_{ii}}$.

**Exercise D.** Store the number of sets of six meters that contain the main meter and leave no hidden attack in `n_d`.

In [ ]:
n_d = None   # your answer

check("Exercise D", n_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_d = sum(12 in P for P in SECURE6)

check("Exercise D", n_d)

In [ ]:
# Figure: the share of protected sets without a hidden attack, and the poll cycle with and without tags
share = [sum(not hidden_possible(P) for P in itertools.combinations(range(13), k)) / math.comb(13, k)
         for k in range(1, 14)]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].bar(range(1, 14), [100 * s for s in share], color="#1E6FA8")
ax[0].set_xticks(range(1, 14))
ax[0].set_xlabel("number of protected meters")
ax[0].set_ylabel("sets without a hidden attack (%)")
bauds = [9600, 19200, 38400, 115200]
ax[1].bar(np.arange(4) - 0.2, [sum(poll_time(n, b) for n in BUS) for b in bauds], 0.4, color="#9DB7CC",
          label="plain frames")
ax[1].bar(np.arange(4) + 0.2, [sum(poll_time(n, b, 12) for n in BUS) for b in bauds], 0.4, color="#1E6FA8",
          label="counter and tag")
ax[1].set_xticks(range(4))
ax[1].set_xticklabels([str(b) for b in bauds])
ax[1].set_xlabel("baud rate")
ax[1].set_ylabel("poll cycle (ms)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe and reflect.** Which defence would you build first on the campus, and why? What do the tags cost the real-time tasks of Week 7?

## 5. The campus microgrid, stage 11

*Lecture: [The campus microgrid, stage 11: Attacks and defences](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#the-campus-microgrid-stage-11)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "hidden_possible" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-11/"
               "NB11_cps_security.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 11 collects what the attacks of this week do to the campus and what the defences cost. The numbers come from the functions of Sections 1 to 4.

In [ ]:
STAGE11 = {
    "import at 19:00 with the rule of Week 1 (kW)": round(float(NET_KW[19] - RULE_KW[19]), 2),
    "import at 19:00 after the forged command (kW)": round(float(NET_KW[19] - BAT_FORGED[19]), 2),
    "state estimate without an attack: J": round(base["J"], 2),
    "one meter: change needed to shed B3 (kW)": round(float(need), 2),
    "one meter: J of that change": round(one["J"], 2),
    "hidden attack of +50 kW: J": round(hidden["J"], 2),
    "hidden attack of +50 kW: B3 estimated (kW)": round(float(hidden["x"][2]), 2),
    "fewest protected meters without a hidden attack": next(
        k for k in range(1, 14) if any(not hidden_possible(P) for P in itertools.combinations(range(13), k))),
    "largest hidden shift of B3, B3 feeder meter protected (kW)": round(base["r"][2] + math.sqrt(THRESHOLD * OMEGA[2]), 2),
    "replay without a watermark: trip of the island (s)": island(0, 10)["trip"],
    "replay, watermark 2 kW, window 30 s: alarm (s)": island(2, 30)["alarm"],
    "frequency jitter without a watermark (mHz)": island(0, 10)["jitter_mhz"],
    "frequency jitter with the watermark of 2 kW (mHz)": island(2, 30)["jitter_mhz"],
    "frames of the attacker accepted by the inverter": ATTACKER_ACCEPTED,
    "poll cycle at 19200 baud with counter and tag (ms)": round(SECURE_MS, 2),
}
for key, value in STAGE11.items():
    print(f"{key:<62} {value}")

**Python note.** The call `next(...)` returns the first size of a protected set for which at least one set leaves no hidden attack. The search runs through all 8191 sets of meters, which takes about a second.

In [ ]:
# Figure: the estimate of B3 under the attacks, and the island under the replay
plain, marked = island(0, 10), island(2, 30)
secs = lambda r: np.arange(len(r["f"])) * DT
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.25]})
cases = ["no attack", "one meter", "hidden attack", "hidden attack,\nsix meters protected"]
values = [base["x"][2], one["x"][2], hidden["x"][2], base["x"][2]]
alarms = ["no alarm", "bad-data alarm", "no alarm: B3 shed", "authentication alarm"]
ax[0].bar(range(4), values, color=["#9DB7CC", "#C98510", "#B53F2E", "#2E8B57"])
for k, (v, txt) in enumerate(zip(values, alarms)):
    ax[0].text(k, v + 4, txt, ha="center", fontsize=7.5)
ax[0].axhline(SHED_KW, color="#B53F2E", ls=":", lw=1)
ax[0].set_xticks(range(4))
ax[0].set_xticklabels(cases, fontsize=8)
ax[0].set_ylim(0, 185)
ax[0].set_ylabel("estimate of B3 (kW)")
ax[1].plot(secs(plain), plain["f"], color="#B53F2E", lw=1.2, label="replay, no watermark")
ax[1].plot(secs(marked), marked["f"], color="#1E6FA8", lw=1.2, label="replay, watermark 2 kW")
ax[1].axvline(60, color="#56616D", ls=":", lw=1)
ax[1].axvline(80, color="#56616D", ls=":", lw=1)
ax[1].set_ylim(-2.6, 0.3)
ax[1].set_xlabel("time (s)")
ax[1].set_ylabel("frequency deviation (Hz)")
ax[1].legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

**Exercise E.** Store the largest hidden shift of B3 in kW with the B3 feeder meter protected, as kept in `STAGE11`, in `s_e`.

In [ ]:
s_e = None   # your answer

check("Exercise E", s_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
s_e = STAGE11["largest hidden shift of B3, B3 feeder meter protected (kW)"]

check("Exercise E", s_e)

**Observe and reflect.** Which result of stage 11 would convince the facility manager to pay for protected meters, and which one to pay for tags on the bus?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE11" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-11/"
               "NB11_cps_security.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Wherever a linear model $\mathbf{z} = H\mathbf{x} + \mathbf{e}$ links meters to quantities, an attacker who knows $H$ can hide an injection $\mathbf{a} = H\mathbf{c}$. The function of this section estimates the quantities of a system of your choice and builds the hidden attack on one of them. It then searches for the smallest sets of meters that would stop it.

The systems and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "water"
# choose: water, traffic, battery_pack, hvac, factory, tram
APPS = {  # description, unit, quantities, meters, H, sigma of the meters, true values, target, limit, consequence
    "water": ("the inflows of three districts of a water network", "m3/h", ["district A", "district B", "district C"],
              ["meter A", "meter B", "trunk main B+C", "reservoir outflow"],
              [[1, 0, 0], [0, 1, 0], [0, 1, 1], [1, 1, 1]], [2, 2, 3, 4], [120, 95, 60], 2, 90,
              "the leak alarm closes the valve of district C"),
    "traffic": ("the vehicles per hour on three roads into a city centre", "vehicles/h", ["road 1", "road 2", "road 3"],
                ["counter road 1", "counter road 2", "counter road 3", "ring road camera (1+2)", "toll gate (all)"],
                [[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0], [1, 1, 1]], [20, 20, 20, 30, 40], [640, 720, 410], 1, 900,
                "the traffic lights switch to the congestion plan"),
    "battery_pack": ("the voltages of four cells of a battery module", "V", ["cell 1", "cell 2", "cell 3", "cell 4"],
                     ["cell 1", "cell 2", "cell 3", "cell 4", "cells 1+2", "cells 3+4", "module"],
                     [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [1, 1, 0, 0], [0, 0, 1, 1], [1, 1, 1, 1]],
                     [0.005] * 4 + [0.01, 0.01, 0.02], [3.92, 3.95, 3.93, 3.94], 2, 4.2,
                     "the overvoltage protection opens the contactor of the pack"),
    "hvac": ("the temperatures of three zones of an office floor", "degrees C", ["zone 1", "zone 2", "zone 3"],
             ["zone 1, sensor a", "zone 1, sensor b", "zone 2, sensor a", "zone 2, sensor b", "zone 3, sensor a",
              "zone 3, sensor b", "return air (mean)"],
             [[1, 0, 0], [1, 0, 0], [0, 1, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1], [1 / 3, 1 / 3, 1 / 3]],
             [0.3, 0.5, 0.3, 0.5, 0.3, 0.5, 0.4], [23.0, 24.0, 22.5], 1, 27.0, "the chiller runs at full power"),
    "factory": ("the output of three machines of a production line", "parts/h", ["machine 1", "machine 2", "machine 3"],
                ["counter 1", "counter 2", "counter 3", "conveyor after 1 and 2", "packing station (all)"],
                [[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0], [1, 1, 1]], [2, 2, 2, 3, 4], [118, 104, 96], 2, 130,
                "machine 3 is stopped for an overspeed check"),
    "tram": ("the power of three substations of a tram line", "kW", ["substation 1", "substation 2", "substation 3"],
             ["meter 1", "meter 3", "feeder 1-2", "feeder 2-3", "utility meter (all)"],
             [[1, 0, 0], [0, 0, 1], [1, 1, 0], [0, 1, 1], [1, 1, 1]], [5, 5, 8, 8, 10], [610, 740, 520], 1, 900,
             "the trams of section 2 are slowed down"),
}
def attack_report(app):
    """Estimates the quantities of one system, builds the hidden attack on its target and searches for protection."""
    desc, unit, states, meters, H, sig, x_true, target, limit, consequence = APPS[app]
    H, sig, x_true = np.array(H, float), np.array(sig, float), np.array(x_true, float)
    m, n = H.shape
    W = np.diag(1 / sig ** 2)
    K = np.linalg.solve(H.T @ W @ H, H.T @ W)                   # turns readings into estimates
    z = H @ x_true + sig * np.random.default_rng(5).normal(0, 1, m)
    test = lambda zz: float((zz - H @ (K @ zz)) @ W @ (zz - H @ (K @ zz)))
    x0, thr = K @ z, chi2.ppf(0.99, m - n)
    print(f"{app}: {desc} ({unit})")
    print(f"  estimate {np.round(x0, 3)}, J = {test(z):.2f} against the threshold {thr:.2f}")
    c = np.zeros(n)
    c[target] = 1.01 * limit - x0[target]                      # just above the limit
    a = H @ c
    print(f"  goal of the attacker: {states[target]} above {limit} {unit}, so that {consequence}")
    print(f"  hidden attack: change {[meters[i] for i in np.flatnonzero(np.abs(a) > 1e-12)]} by "
          f"{np.round(a[np.abs(a) > 1e-12], 3)}; J = {test(z + a):.2f}, estimate {(K @ (z + a))[target]:.3f}")
    best = int(np.argmax(np.abs(K[target])))                    # the meter with the largest share in the target
    single = np.zeros(m)
    single[best] = c[target] / K[target, best]
    print(f"  one meter: change {meters[best]} by {single[best]:.3f}; J = {test(z + single):.2f}")
    no_hidden = lambda P: np.linalg.matrix_rank(H[list(P)]) == n
    target_safe = lambda P: np.linalg.matrix_rank(np.vstack([H[list(P)], np.eye(n)[target]])) ==         np.linalg.matrix_rank(H[list(P)])
    sets = [P for k in range(1, m + 1) for P in itertools.combinations(range(m), k)]
    print("  no hidden attack at all with the protected meters", [meters[i] for i in next(filter(no_hidden, sets))])
    print(f"  no hidden attack on {states[target]} with the protected meters",
          [meters[i] for i in next(filter(target_safe, sets))])

attack_report(APP)

**Python note.** The function `attack_report` keeps its variables inside, so the names of the earlier sections stay untouched. The test `target_safe` asks whether the target can be computed from the protected meters alone. Adding the unit vector of the target to the protected rows of $H$ must then leave the rank unchanged. Every hidden change of the target would change a protected meter. A meter that no combination of the other meters can replace is called critical: Its residual is always 0, and the test cannot see any change of it.

**Observe and reflect.** In which applications does a single protected meter stop the hidden attack on the target, and why do the others need two? Why does a change of one meter stay hidden in the water network?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write the authenticated frame of Section 4 as a small class with the methods `pack` and `unpack`, and test that a change of any single byte is rejected. |
| Embedded systems | Measure how long one HMAC tag with SHA-256 takes on your computer. Estimate whether a microcontroller of 48 MHz could tag every frame of the bus in time. |
| Internet of Things | Design MQTT messages (Week 8) for the chargers of the car park that carry a counter and a tag, and test them against a replayed message. |
| Computer networks | The attacker blocks the frames of the frequency meter instead of replaying them. Add a watchdog of 0.5 s to `island` and decide what the battery should do when it fires. |
| Control engineering | Replace the fixed threshold of g by the CUSUM of Week 10 and compare the time to detection and the false alarms. |
| Data science | Simulate 500 sets of readings with and without hidden attacks of random size and train a classifier on them. Which attacks does it catch that the chi-square test misses? |
| Cybersecurity | The main meter is protected. Find the smallest number of meters that an attacker must control to raise the B3 estimate by 50 kW without an alarm, and list them. |
| Artificial intelligence | Train a small network to tell replayed windows of the frequency from live ones. Then change a replayed window as little as possible until the network misses it. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-11/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-11).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>